In [1]:
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score, classification_report

# **Final Project: Translation Detection**

**Motivation**: Knowing that a text is a translation is important since the author’s original intentions can become lost in translation. As a reader, if you know that a text is originally written in the language or if it is the result of a translation can increase your awareness and change your interpretation of the text.
Different languages carry many subtle differences that can be hard for humans to perceive, so translations between languages may carry some traces of these language differences and carry patterns that can be detected by machine learning techniques.

**Research Question**: Can NLP algorithms accurately classify whether a text was originally written in English or translated into English from a different language?

In [2]:
meta = pd.read_csv("Complete_Data/TransComp_metadata.csv")
original_eng = pd.read_csv("Complete_Data/Original_samples.csv")
translated = pd.read_csv("Complete_Data/Translation_samples.csv")

#The total num of data (English & Non-English docs) available for us to test
eng_count = (meta['final_lang'] == 'eng').sum()
non_eng_count = (meta['final_lang'] != 'eng').sum()

print("Number of English docs:", eng_count)
print("Number of Non-English docs", non_eng_count)

#The number of features for a document
features_count_eng = (original_eng['docid'] == 'mdp.39015020852250').sum()
features_count_non_eng = (translated['docid'] == 'coo.31924006114809').sum()

print("Number of features for for English doc 'mdp.39015020852250':", features_count_eng)
print("Number of features for for non-English doc 'coo.31924006114809':", features_count_non_eng)


Number of English docs: 10683
Number of Non-English docs 10630
Number of features for for English doc 'mdp.39015020852250': 2668
Number of features for for non-English doc 'coo.31924006114809': 2358


Since we have around 10,000 documents and 2,000 features for each, we have over 20,000,000 rows/features. This can be computationally expensive to process, so we will need to reduce the number of features by keeping the top most frequent words. Doing this carefully will not affect our analysis much because frequent words are able to capture the main stylistic and lexical differences between english language and translated text. However, we will need to be careful not to over-reduce because removing too many features may cause us to lost important discriminative patterns.  

The number of rows/features is still very large. The next option is to reduce the data (document) we have. (Note: usually we want to avoid this because the more data, the better. The model will see more example and learn patterns that help it classify unseen data.)

In [ ]:
#Adjust as needed
num_data = 150
top_words = 500

#Keep top n (num_data) documents 
kept_eng_docs = original_eng['docid'].unique()[:num_data]
kept_trans_docs = translated['docid'].unique()[:num_data]

#For each document, keep top n (top_words) frequent words
reduced_eng_data = original_eng[original_eng['docid'].isin(kept_eng_docs)].groupby('docid').head(top_words)
reduced_trans_data = translated[translated['docid'].isin(kept_trans_docs)].groupby('docid').head(top_words)

#Save as new csv file
reduced_eng_data.to_csv("Reduced_Data/Reduced_original_doc.csv", index=False)
reduced_trans_data.to_csv("Reduced_Data/Reduced_translated_doc.csv", index=False)

Concatenate the reduced original (English) and translated files into one big csv file:

In [ ]:
orig = pd.read_csv("Reduced_Data/Reduced_original_doc.csv")
trans = pd.read_csv("Reduced_Data/Reduced_translated_doc.csv")

#Add labels -> 0 = original & 1 = translated
orig['label'] = 0
trans['label'] = 1

#Combine datasets
data = pd.concat([orig, trans], ignore_index=True)
data.to_csv("Reduced_Data/Concatenated_data.csv", index=False)

### Multinomial Naive Bayes
**More Info:** https://www.geeksforgeeks.org/machine-learning/multinomial-naive-bayes/#:~:text=Multinomial%20Naive%20Bayes%20classifies%20text,as%20spam%20or%20not%20spam. 

In [27]:
#Dictionary of docid with its terms and count
doc_dicts = data.groupby('docid').apply(lambda x: x.set_index('term')['count'].to_dict()).tolist()
#Label for each document
y = data.groupby("docid")["label"].first().values

vectorizer = DictVectorizer(sparse=True)
X = vectorizer.fit_transform(doc_dicts)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = MultinomialNB()
model.fit(X_train, y_train)

#Get predictions and probabilities on held out data
y_pred = model.predict(X_test)
y_pred_probability = model.predict_proba(X_test)

#Model Performance Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision and Recall for each class:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8666666666666667
Precision and Recall for each class:
              precision    recall  f1-score   support

           0       0.85      0.85      0.85        27
           1       0.88      0.88      0.88        33

    accuracy                           0.87        60
   macro avg       0.87      0.87      0.87        60
weighted avg       0.87      0.87      0.87        60



In [ ]:
#Top-20 highest weighted words for each category
print("Top-20 Highest Weighted Words for Each Category:")
feature_names = vectorizer.get_feature_names_out()
for i in range(len(model.classes_)):
    label = model.classes_[i]
    log_probs = model.feature_log_prob_[i]
    top10_indices = log_probs.argsort()[-20:][::-1]
    doc_type = "Original - English" if label == 0 else "Translated"
    print(f"\nCategory: {label} ({doc_type})")
    for idx in top10_indices:
        print(f"  - {feature_names[idx]}")


Top-20 Highest Weighted Words for Each Category:

Category: 0 (Original - English)
  - the
  - and
  - to
  - a
  - of
  - i
  - he
  - in
  - was
  - it
  - you
  - that
  - she
  - his
  - her
  - s
  - had
  - with
  - for
  - on

Category: 1 (Translated)
  - the
  - and
  - to
  - of
  - a
  - in
  - i
  - he
  - was
  - that
  - it
  - you
  - his
  - her
  - had
  - she
  - with
  - s
  - on
  - for


### Logistic Regression